### Disclaimer
I noted that in dataset used to create the prompts there is a issue: proc_text and dose_text are duplicated, since in the check_data_createin_evo.ipynb I found a typo in the code. Now it's time to fix that

In [21]:
# importing libraries and data
import pandas as pd
import ast
import numpy as np

test_df = pd.read_csv("/root/MIMICIV/data/splitted/landmark_evo_test_dod.csv")
train_df = pd.read_csv("/root/MIMICIV/data/splitted/landmark_evo_train_dod.csv")
vali_df = pd.read_csv("/root/MIMICIV/data/splitted/landmark_evo_vali_dod.csv")

df = pd.read_csv("/root/MIMICIV/src/mimiciv_clinical_dataset_tabular_death_visit_evo.csv")

<positron-console-cell-21>:6: DtypeWarning: Columns (16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
<positron-console-cell-21>:7: DtypeWarning: Columns (16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.


In [22]:
def safe_join(x):
    if pd.isna(x) or x in ['[]', '', 'None', 'NaN', 'na', 'nan']:
        return ''
    try:
        x_eval = ast.literal_eval(x)
        if isinstance(x_eval, list):
            return '\n'.join(x_eval)
        else:
            return ''
    except:
        return ''

landmark_rows = []

#df = df.sort_values(['subject_id', 'hadm_id']).reset_index(drop=True) # That's not the right ordering!
# the dataset is still ordered, however we should consider a more robust way of doing so.

# Okay, in *_df we have the error. df instead has the right information. Let's proceed to fix that
landmark_rows = []

for patient_id, group in df.groupby('subject_id'):
    group = group.reset_index(drop=True)
    n_visits = group.shape[0]

    group['medication_list'] = group['medication_list'].apply(safe_join)
    group['diagnosis_list'] = group['diagnosis_list'].apply(safe_join)
    group['procedure_list'] = group['procedure_list'].apply(safe_join)
    group['dose_list'] = group['dose_list'].apply(safe_join)

    for landmark_idx in range(n_visits):
        current_visit = group.iloc[landmark_idx]

        all_visit_so_far = group.iloc[:landmark_idx + 1]

        current_medications = set(current_visit['medication_list'].split('\n'))
        current_diagnoses = set(current_visit['diagnosis_list'].split('\n'))
        current_procedures = set(current_visit['procedure_list'].split('\n'))
        current_dose = set(current_visit['dose_list'].split('\n'))

        if landmark_idx > 0:
            # Get past visits
            past_visits = group.iloc[(landmark_idx-1):landmark_idx]
            past_medications = set("\n".join(past_visits['medication_list']).split('\n'))
            past_diagnoses = set("\n".join(past_visits['diagnosis_list']).split('\n'))
            past_procedures = set("\n".join(past_visits['procedure_list']).split('\n'))
            past_dose = set("\n".join(past_visits['dose_list']).split('\n'))

            new_medications = current_medications.difference(past_medications)
            new_diagnoses = current_diagnoses.difference(past_diagnoses)
            new_procedures = current_procedures.difference(past_procedures)
            new_dose = current_dose.difference(past_dose)
            no_more_diagnoses = past_diagnoses.difference(current_diagnoses)
            no_more_medications = past_medications.difference(current_medications)
            no_more_procedures = past_procedures.difference(current_procedures)
            no_more_dose = past_dose.difference(current_dose)
        else:
            new_medications = current_medications
            new_diagnoses = current_diagnoses
            new_procedures = current_procedures
            new_dose = current_dose
            no_more_diagnoses = set()
            no_more_medications = set()
            no_more_procedures = set()
            no_more_dose = set()

        # Informations per visit
        # Create dictionaries to store information per visit
        meds_per_visit = {}
        diag_per_visit = {}
        proc_per_visit = {}
        dose_per_visit = {}

        for visit_idx in range(landmark_idx + 1):
            # Get the visit information
            visit = group.iloc[visit_idx]
            # Split the lists into individual items and store them
            # in the dictionaries, Handling NaN values
            meds  = visit['medication_list'].split('\n') if pd.notna(visit['medication_list']) else []
            diags = visit['diagnosis_list'].split('\n')  if pd.notna(visit['diagnosis_list'])  else []
            procs = visit['procedure_list'].split('\n')  if pd.notna(visit['procedure_list'])  else []
            dose  = visit['dose_list'].split('\n')       if pd.notna(visit['dose_list'])       else []
            
            # Store the lists in the dictionaries
            meds_per_visit[visit_idx + 1] = meds
            diag_per_visit[visit_idx + 1] = diags
            proc_per_visit[visit_idx + 1] = procs
            dose_per_visit[visit_idx + 1] = dose

        # Create text representations of the lists
        # Needed for naive and no_narratives prompts
        unique_vals = set(all_visit_so_far['medication_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            meds_text = ''
        else:
            meds_text = '\n'.join(all_visit_so_far['medication_list'])

        unique_vals = set(all_visit_so_far['diagnosis_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            diag_text = ''
        else:
            diag_text = '\n'.join(all_visit_so_far['diagnosis_list'])

        unique_vals = set(all_visit_so_far['procedure_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            proc_text = ''
        else:
            proc_text = '\n'.join(all_visit_so_far['procedure_list'])
        
        unique_vals = set(all_visit_so_far['dose_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            dose_text = ''
        else:
            dose_text = '\n'.join(all_visit_so_far['dose_list'])

        days_to_death = current_visit['days_from_last_visit_to_death']
        death_in_90days = 1 if (not np.isnan(days_to_death) and days_to_death <= 90) else 0

        landmark_rows.append({
            'subject_id': current_visit['subject_id'],
            'hadm_id': current_visit['hadm_id'],
            'admission_category': current_visit['admission_category'],
            'landmark_visit': landmark_idx + 1,
            'age_at_landmark': current_visit['age_at_event'],
            'gender': current_visit['gender'],
            'num_total_visits': n_visits,  # No used
            'days_from_last_visit': current_visit['days_until_next_visit'],
            'death_in_90days': death_in_90days,

            'med_text': meds_text,   # naive no_narratives
            'diag_text': diag_text,  # naive no_narratives
            'proc_text': proc_text,  # naive no_narratives
            'dose_text': dose_text,  # naive no_narratives

            'new_medications': new_medications,         # XGBoost
            'new_diagnoses': new_diagnoses,             # XGBoost
            'new_procedures': new_procedures,           # XGBoost
            'new_dose': new_dose,                       # XGBoost
            'no_more_diagnoses': no_more_diagnoses,     # XGBoost
            'no_more_medications': no_more_medications, # XGBoost
            'no_more_procedures': no_more_procedures,   # XGBoost
            'no_more_dose': no_more_dose,               # XGBoost
 
            'meds_per_visit': meds_per_visit,           # full_* and compact_*
            'diag_per_visit': diag_per_visit,           # full_* and compact_*
            'proc_per_visit': proc_per_visit,           # full_* and compact_* 
            'dose_per_visit': dose_per_visit            # full_* and compact_*
        })

landmark_df = pd.DataFrame(landmark_rows)

In [23]:
train_df_fixed = train_df.merge(landmark_df[['subject_id','hadm_id', 'proc_text']], on=['subject_id','hadm_id'], how = 'left', suffixes=('_OLD', '_NEW'))
train_df_fixed['proc_text_OLD'] = train_df_fixed['proc_text_NEW']
train_df_fixed = (
    train_df_fixed
    .drop(columns=['proc_text_NEW'])  # Elimina la colonna nuova ridondante
    .rename(columns={'proc_text_OLD': 'proc_text'})  # Ripristina nome originale
)

In [24]:
vali_df_fixed = vali_df.merge(landmark_df[['subject_id','hadm_id', 'proc_text']], on=['subject_id','hadm_id'], how = 'left', suffixes=('_OLD', '_NEW'))
vali_df_fixed['proc_text_OLD'] = vali_df_fixed['proc_text_NEW']
vali_df_fixed = (
    vali_df_fixed
    .drop(columns=['proc_text_NEW'])  # Elimina la colonna nuova ridondante
    .rename(columns={'proc_text_OLD': 'proc_text'})  # Ripristina nome originale
)

In [25]:
test_df_fixed = test_df.merge(landmark_df[['subject_id','hadm_id', 'proc_text']], on=['subject_id','hadm_id'], how = 'left', suffixes=('_OLD', '_NEW'))
test_df_fixed['proc_text_OLD'] = test_df_fixed['proc_text_NEW']
test_df_fixed = (
    test_df_fixed
    .drop(columns=['proc_text_NEW'])  # Elimina la colonna nuova ridondante
    .rename(columns={'proc_text_OLD': 'proc_text'})  # Ripristina nome originale
)

In [26]:
test_df_fixed.to_csv("/root/MIMICIV/data/splitted/landmark_evo_test_dod_fxd.csv", index=False)
vali_df_fixed.to_csv("/root/MIMICIV/data/splitted/landmark_evo_vali_dod_fxd.csv", index=False)
train_df_fixed.to_csv("/root/MIMICIV/data/splitted/landmark_evo_train_dod_fxd.csv", index=False)
